# Project 3.2 – Basic Machine Learning Implementation

This project uses the UCI Heart Disease dataset to complete classification, regression, and clustering tasks.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv('heart_disease_uci.csv')
df.head()


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


## Data Preparation
The dataset contains both numerical and categorical variables. Missing values are handled and categorical fields are encoded before modeling.

In [2]:
data = df.copy()

categorical_cols = data.select_dtypes(include='object').columns
for col in categorical_cols:
    data[col] = LabelEncoder().fit_transform(data[col].astype(str))

bool_cols = data.select_dtypes(include='bool').columns
for col in bool_cols:
    data[col] = data[col].astype(int)

imputer = SimpleImputer(strategy='median')
data = pd.DataFrame(imputer.fit_transform(data), columns=data.columns)

data.head()


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1.0,63.0,1.0,0.0,3.0,145.0,233.0,1.0,0.0,150.0,0.0,2.3,0.0,0.0,0.0,0.0
1,2.0,67.0,1.0,0.0,0.0,160.0,286.0,0.0,0.0,108.0,1.0,1.5,1.0,3.0,2.0,2.0
2,3.0,67.0,1.0,0.0,0.0,120.0,229.0,0.0,0.0,129.0,1.0,2.6,1.0,2.0,3.0,1.0
3,4.0,37.0,1.0,0.0,2.0,130.0,250.0,0.0,2.0,187.0,0.0,3.5,0.0,0.0,2.0,0.0
4,5.0,41.0,0.0,0.0,1.0,130.0,204.0,0.0,0.0,172.0,0.0,1.4,3.0,0.0,2.0,0.0


## Classification Task

The objective of this task is to predict whether a patient has heart disease using a neural network classifier. Performance will be evaluated using accuracy, precision, recall, and F1 score.


In [3]:
### Classification Findings

The neural network classifier achieved approximately 86% accuracy. Precision, recall, and F1 score indicate that the model successfully identified patients with heart disease while maintaining balanced performance across evaluation metrics.

SyntaxError: invalid syntax (1311102195.py, line 3)

## Classification Task
Predict whether a patient has heart disease. The target variable is converted to binary form where 0 indicates no disease and 1 indicates disease.

In [ ]:
classification_df = data.copy()
classification_df['target'] = (classification_df['num'] > 0).astype(int)

X = classification_df.drop(['num','target'], axis=1)
y = classification_df['target']

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=32,
                    validation_split=0.2,
                    verbose=1)

pred = (model.predict(X_test) > 0.5).astype(int)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


## Regression Task

The objective of this task is to predict cholesterol levels using a neural network regression model.

## Regression Task
Predict cholesterol levels using patient characteristics.

In [ ]:
regression_df = data.copy()

X = regression_df.drop('chol', axis=1)
y = regression_df['chol']

X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

regressor = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

regressor.compile(optimizer='adam',
                  loss='mse',
                  metrics=['mae'])

regressor.fit(X_train, y_train,
              epochs=20,
              batch_size=32,
              validation_split=0.2,
              verbose=1)

pred = regressor.predict(X_test)

mse = mean_squared_error(y_test, pred)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("MSE:", mse)
print("MAE:", mae)
print("R2:", r2)


## Clustering Task

The objective of this task is to identify naturally occurring groups of patients using K-Means clustering.

## Clustering Task
K-Means clustering is used to identify groups of patients with similar health characteristics.

In [ ]:
cluster_features = data[['age','trestbps','chol','thalch','oldpeak']]

scaled = StandardScaler().fit_transform(cluster_features)

inertia = []
for k in range(1,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    inertia.append(km.inertia_)

plt.plot(range(1,11), inertia)
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled)

score = silhouette_score(scaled, clusters)
print('Silhouette Score:', score)


### Clustering Findings

K-Means clustering identified three patient groups. The silhouette score indicated moderate separation between clusters, suggesting meaningful but overlapping patient segments.

## Summary

### Classification
A neural network classifier was developed to predict heart disease. Accuracy, precision, recall, and F1-score were used to evaluate performance.

### Regression
A neural network regressor was used to predict cholesterol levels. Performance was evaluated using MSE, MAE, and R².

### Clustering
K-Means clustering was used to identify natural groupings within the patient population. The elbow method and silhouette score were used for evaluation.
